In [1]:
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams.update({'errorbar.capsize': 3})
plt.rcParams['xtick.labelsize']= 13
plt.rcParams['ytick.labelsize']= 13
plt.rcParams.update({'axes.labelsize': 16})

In [2]:
from astropy.time import Time
from astropy.table import  Table

In [3]:
from astropy import constants, units

c_kms=constants.c.value/10**3

In [4]:
import h5py
from astropy.io import ascii

In [5]:
import pickle

In [6]:
from lmfit import Model

In [7]:

def gaussian_w_slope(x, amp, cen, wid, bas,slope=0):
    "1-d gaussian: gaussian(x, amp, cen, wid,bas,slope)"
    return (amp/(np.sqrt(2*np.pi)*wid)) * np.exp(-(x-cen)**2 /(2*wid**2)) + bas+x*slope

In [8]:
def wl2vel(wl_arr,central_wl=None):
    '''returns vel in km/s
    
        inputs are wl_arr and central_wl'''
    if central_wl==None:
        central_wl=np.average(wl_arr)

    #print(c_kms)
    vel_arr=(wl_arr-central_wl)*c_kms/central_wl
    return vel_arr

In [9]:
#read coadds from coadds_outputs.h5 

data=h5py.File('coadds_outputs_2024.h5','r')

In [10]:
#find at which indices there are data for both NEID coadds and EXPRES coadds
neid_indices=[]
expres_indices=[]
overlap_mids=[]

for i,key in  enumerate(data['NEID']['coadds'].keys()):
    #check if there is data for that key in NEID that is not empty
    if data['NEID']['coadds'][key] is not None and len(data['NEID']['coadds'][key]) > 0:
        print('NEID coadds found for key:', key)
        #check if that key is also in EXPRES coadds
        if key in data['EXPRES']['coadds'].keys():
            #check if there is data for that key in EXPRES that is not empty
            if data['EXPRES']['coadds'][key] is not None and len(data['EXPRES']['coadds'][key]) > 0:
                print('EXPRES coadds found for key:', key)
                neid_indices.append(key)
                expres_indices.append(key)
                overlap = data['overlaps'][i]
                mid=(overlap[0]+overlap[1])/2
                overlap_mids.append(Time(mid,format='jd').isot)

NEID coadds found for key: 0000
EXPRES coadds found for key: 0000
NEID coadds found for key: 0001
EXPRES coadds found for key: 0001
NEID coadds found for key: 0002
EXPRES coadds found for key: 0002
NEID coadds found for key: 0003
EXPRES coadds found for key: 0003
NEID coadds found for key: 0004
EXPRES coadds found for key: 0004
NEID coadds found for key: 0005
EXPRES coadds found for key: 0005
NEID coadds found for key: 0006
EXPRES coadds found for key: 0006
NEID coadds found for key: 0007
EXPRES coadds found for key: 0007
NEID coadds found for key: 0008
EXPRES coadds found for key: 0008
NEID coadds found for key: 0009
EXPRES coadds found for key: 0009
NEID coadds found for key: 0010
EXPRES coadds found for key: 0010
NEID coadds found for key: 0011
EXPRES coadds found for key: 0011
NEID coadds found for key: 0012
EXPRES coadds found for key: 0012
NEID coadds found for key: 0013
EXPRES coadds found for key: 0013
NEID coadds found for key: 0014
EXPRES coadds found for key: 0014
NEID coadd

In [11]:
data['overlaps']

<HDF5 dataset "overlaps": shape (61, 2), type "<f8">

In [12]:
#check neid_indices = expres_indices and that their length is the same as overlap_mids
if neid_indices == expres_indices and len(neid_indices) == len(overlap_mids):

    print('everything matches:', neid_indices)
else:
    print('NEID and EXPRES indices do not match')

everything matches: ['0000', '0001', '0002', '0003', '0004', '0005', '0006', '0007', '0008', '0009', '0010', '0011', '0012', '0013', '0014', '0015', '0016', '0017', '0018', '0019', '0020', '0021', '0023', '0024', '0025', '0026', '0027', '0028', '0029', '0030', '0031', '0032', '0033', '0034', '0035', '0036', '0037', '0038', '0039', '0040', '0041', '0042', '0043', '0044', '0045', '0046', '0047', '0048', '0049', '0050', '0051', '0052', '0053', '0054', '0055', '0056', '0057', '0058', '0059', '0060']


In [13]:
print(len(neid_indices), len(overlap_mids))

60 60


In [14]:
#read in csv file with line list
line_list_loc="../../RvLineList_allowBlends=0_overlapCutoff=5.0e-6_blend_RV_cutoff=2.0_depthCutoff=0.02_rejectTelluricSlope=2000.0_badLineFilter=none_quant=95_nbin=1_DP=true_binParam=depth_n=1_long_output=true_VACUUM.csv"

In [15]:
line_list=ascii.read(line_list_loc)

In [16]:
gmod=Model(gaussian_w_slope)

In [17]:


def fit_lines(w,s,line_list,plot=False,wl_min=6000,wl_max=6030,fit_range=0.7,dist_from_edge=500):
    line_fits=[]
    '''fits lines in line_list to the spectrum s with wavelengths w
    
        inputs are:
            w: 2D array of wavelengths (orders x pixels)
            s: 2D array of spectrum (orders x pixels)
            line_list: astropy table with columns 'lambda' and 'depth'
    '''
    for i,l in enumerate(line_list['lambda'].value):
        if l<wl_max and l>wl_min:
            #print(l,line_list['depth'][i])
            #find all the orders that contain this wavelength, and the line is more than 500 pixels from the edge of the order  
            orders_with_line=[] 

            for o in range(w.shape[0]):
                if l>w[o].min()+dist_from_edge*(w[o][1]-w[o][0]) and l<w[o].max()-dist_from_edge*(w[o][1]-w[o][0]):
                    orders_with_line.append(o)
            #print("orders with line:",orders_with_line)
            #for each order, convert the wavelength to velocity, centered on the line wavelength and then fit a guassian to the spectrum
            #only fit within 2 angstroms of the line center
            for o in orders_with_line: 
                wl_order=w[o]
                s_order=s[o]
                vel_order=wl2vel(wl_order,central_wl=l)
                #select only the points within 1 angstroms of the line center and the spectrum is not NaN
                mask=(wl_order>l-fit_range)&(wl_order<l+fit_range)&(~np.isnan(s_order))
                wl_fit=wl_order[mask]
                s_fit=s_order[mask]
                vel_fit=vel_order[mask]
                #save the pixel index of the line center
                line_center_index=np.argmin(np.abs(wl_order-l))
                #initial guess for the gaussian parameters
                amp_init=line_list['depth'][i]*np.sum(s_fit)*-1
                cen_init=0
                wid_init=3.0
                bas_init=np.median(s_fit)
                slope_init=0
                params=gmod.make_params(amp=amp_init,cen=cen_init,wid=wid_init,bas=bas_init,slope=slope_init)
                #fit the gaussian to the data
                try:
                    result=gmod.fit(s_fit,params,x=vel_fit)
                    #now refit after dividing by the best fit baseline + slope
                    s_fit_detrended=s_fit/(result.params['bas'].value+vel_fit*result.params['slope'].value)
                    params=gmod.make_params(amp=amp_init,cen=cen_init,wid=wid_init,bas=0,slope=0)
                    result=gmod.fit(s_fit_detrended,params,x=vel_fit)
                
                except:
                    print(f"Fit failed for line at {l} Angstroms in order {o}")
                    print(wl_fit)
                    print(s_fit)
                    continue               
                #print(result.fit_report())
                if plot:
                    #plot the data and the fit
                    plt.figure()
                    plt.plot(vel_fit,s_fit_detrended,label='data')
                    plt.plot(vel_fit,result.best_fit,label='fit')
                    plt.xlabel('Velocity (km/s)')
                    plt.ylabel('Flux')
                    plt.title(f'Order {o} Line at {l} Angstroms')
                    plt.legend()
                    plt.show()
                
                

                
                #save the fit results to a list
                line_fits.append({'line_wl':l,'order':o,'amp':result.params['amp'].value,'amp_err':result.params['amp'].stderr,    
                                  'cen':result.params['cen'].value,
                                  'cen_err':result.params['cen'].stderr,
                                  'wid':result.params['wid'].value,
                                  'wid_err':result.params['wid'].stderr,
                                  'bas':result.params['bas'].value,
                                  'bas_err':result.params['bas'].stderr,
                                  'slope':result.params['slope'].value,'chi2':result.chisqr,
                                  'bic':result.bic,'r':result.rsquared,
                                  'line_center_index':line_center_index})
    
    return line_fits
            

In [18]:
dist_from_edge=500

In [19]:
line_list['lambda'].value

array([3882.13403658, 3882.72724838, 3884.7318502 , ..., 9802.98268077,
       9864.427174  , 9891.73905874])

In [20]:
fit_results=[]
#loop through the overlapping indices and fit lines to the NEID and EXPRES coadds
for i,idx in enumerate(neid_indices[:2]):

    w_NEID=np.array(data['NEID']['wls'][idx])
    s_NEID=np.array(data['NEID']['coadds'][idx])

    w_EXPRES=np.array(data['EXPRES']['wls'][idx])
    s_EXPRES=np.array(data['EXPRES']['coadds'][idx])

    upper=np.min((np.max(w_NEID),np.max(w_EXPRES)))
    lower=np.max((np.min(w_NEID),np.min(w_EXPRES)))

    #find line fits in both spectra that are at least 500 pixels from the edge of at least one order
    lines_to_use={}

    cols_to_keep=['lambda','depth']
    for col in cols_to_keep:
        lines_to_use[col]=[]
    for row in line_list:
        l=row['lambda']
    
        if l<upper and l>lower:
            #find all the orders that contain this wavelength, and the line is more than 500 pixels from the edge of the order  
            #if line is not atleast 500 pixels from the edge of at least one order in both NEID and EXPRES, include it in lines_to_use
            neid_ok=False
            expres_ok=False
            for o in range(w_NEID.shape[0]):
                if l>w_NEID[o].min()+dist_from_edge*(w_NEID[o][1]-w_NEID[o][0]) and l<w_NEID[o].max()-dist_from_edge*(w_NEID[o][1]-w_NEID[o][0]):
                    neid_ok=True
            for o in range(w_EXPRES.shape[0]):
                if l>w_EXPRES[o].min()+dist_from_edge*(w_EXPRES[o][1]-w_EXPRES[o][0]) and l<w_EXPRES[o].max()-dist_from_edge*(w_EXPRES[o][1]-w_EXPRES[o][0]):
                    expres_ok=True
            if neid_ok and expres_ok:
                for col in cols_to_keep:
                    lines_to_use[col].append(row[col])
    lines_to_use=Table(lines_to_use)

    #print number of fitting lines for index idx
    #print(lines_to_use)
    print(f"Fitting {len(lines_to_use)} lines for index {idx} between {lower:.1f} and {upper:.1f} Angstroms")
    print("Now fitting for NEID...")
    line_fits_NEID=fit_lines(w_NEID,s_NEID,lines_to_use,plot=False,wl_min=lower,wl_max=upper,fit_range=0.7,dist_from_edge=500)
    print("Now fitting for EXPRES...")
    line_fits_EXPRES=fit_lines(w_EXPRES,s_EXPRES,lines_to_use,plot=False,wl_min=lower,wl_max=upper,fit_range=0.7,dist_from_edge=500)
    fit_results.append({'index':idx,'NEID_fits':line_fits_NEID,'EXPRES_fits':line_fits_EXPRES,'overlap_mid':overlap_mids[i]})
    #

Fitting 1502 lines for index 0000 between 3796.7 and 8223.6 Angstroms
Now fitting for NEID...
Now fitting for EXPRES...
Fit failed for line at 3882.134036576609 Angstroms in order 3
[]
[]


/opt/anaconda3/envs/astro/lib/python3.13/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/anaconda3/envs/astro/lib/python3.13/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Fit failed for line at 3897.5725597378896 Angstroms in order 2
[]
[]
Fit failed for line at 3899.4873657988883 Angstroms in order 4
[]
[]
Fit failed for line at 3900.4830103985014 Angstroms in order 4
[]
[]
Fit failed for line at 3900.8139454679713 Angstroms in order 4
[]
[]
Fit failed for line at 3901.319662629716 Angstroms in order 4
[]
[]
Fit failed for line at 3902.0609527015436 Angstroms in order 4
[]
[]
Fit failed for line at 3902.6934833013465 Angstroms in order 4
[]
[]
Fit failed for line at 3905.177458248997 Angstroms in order 4
[]
[]
Fit failed for line at 3921.370473401626 Angstroms in order 3
[]
[]
Fit failed for line at 3948.1093100410044 Angstroms in order 4
[]
[]
Fit failed for line at 3950.2481908490395 Angstroms in order 6
[]
[]
Fit failed for line at 3951.070065911864 Angstroms in order 6
[]
[]
Fit failed for line at 3951.464159811495 Angstroms in order 6
[]
[]
Fit failed for line at 3952.7327563899594 Angstroms in order 6
[]
[]
Fit failed for line at 3953.30433625950

/opt/anaconda3/envs/astro/lib/python3.13/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,


Fit failed for line at 3897.5725597378896 Angstroms in order 2
[]
[]
Fit failed for line at 3899.4873657988883 Angstroms in order 4
[]
[]
Fit failed for line at 3900.4830103985014 Angstroms in order 4
[]
[]
Fit failed for line at 3900.8139454679713 Angstroms in order 4
[]
[]
Fit failed for line at 3901.319662629716 Angstroms in order 4
[]
[]
Fit failed for line at 3902.0609527015436 Angstroms in order 4
[]
[]
Fit failed for line at 3902.6934833013465 Angstroms in order 4
[]
[]
Fit failed for line at 3905.177458248997 Angstroms in order 4
[]
[]
Fit failed for line at 3921.370473401626 Angstroms in order 3
[]
[]
Fit failed for line at 3948.1093100410044 Angstroms in order 4
[]
[]
Fit failed for line at 3950.2481908490395 Angstroms in order 6
[]
[]
Fit failed for line at 3951.070065911864 Angstroms in order 6
[]
[]
Fit failed for line at 3951.464159811495 Angstroms in order 6
[]
[]
Fit failed for line at 3952.7327563899594 Angstroms in order 6
[]
[]
Fit failed for line at 3953.30433625950

In [23]:
#save fit results to pickele file

with open('line_fit_results_neid_expres_overlap.pkl','wb') as f:
    pickle.dump(fit_results,f)